In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Conv2D, MaxPooling2D, Flatten, Input
import xgboost as xgb
from sklearn.model_selection import train_test_split

print("Libraries imported successfully. TensorFlow version:", tf.__version__)

Libraries imported successfully. TensorFlow version: 2.20.0


In [ ]:
IS_COLAB = 'google.colab' in str(get_ipython())
print(f"Running in Colab: {IS_COLAB}")

Running in Colab: True


In [ ]:
# Install py7zr to handle .7z archives
%pip install py7zr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 16.3 MB/s eta 0:00:00


In [ ]:
import h5py
import py7zr

# Mount Google Drive if in Colab to access the dataset
if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive', force_remount=True)

    gdrive_7z_path = '/content/drive/MyDrive/TCIR-CPAC_IO_SH.7z'
    extracted_dir = '/content/tcir_extracted'
    os.makedirs(extracted_dir, exist_ok=True)

    print(f"Attempting to extract TCIR Dataset from: {gdrive_7z_path}...")
    try:
        try:
            import py7zr
        except ImportError:
            print("Installing py7zr...")
            %pip install py7zr
            import py7zr

        # Skip re-extracting if it's already been done (extraction itself
        # is slow and disk-bound, no need to repeat it every run)
        existing_h5 = None
        for root, dirs, files in os.walk(extracted_dir):
            for file in files:
                if file.endswith('.h5'):
                    existing_h5 = os.path.join(root, file)
                    break
            if existing_h5:
                break

        if existing_h5:
            print(f"Found already-extracted file, skipping extraction: {existing_h5}")
            tcir_path = existing_h5
        else:
            with py7zr.SevenZipFile(gdrive_7z_path, mode='r') as z:
                z.extractall(path=extracted_dir)
            print(f"TCIR .7z archive extracted to: {extracted_dir}")

            tcir_path = None
            for root, dirs, files in os.walk(extracted_dir):
                for file in files:
                    if file.endswith('.h5'):
                        tcir_path = os.path.join(root, file)
                        break
                if tcir_path:
                    break

        if not tcir_path:
            raise FileNotFoundError(f"No .h5 file found in {extracted_dir} after extraction.")

    except Exception as e:
        print(f"Error during extraction: {e}")
        tcir_path = None
# else:
#     print("Assuming TCIR-CPAC_IO_SH.7z is manually extracted in Jupyter environment.")
#     tcir_path = 'TCIR_dataset.h5'

# ---- Inspect the file WITHOUT loading the image matrix ----
if tcir_path and os.path.exists(tcir_path):
    print(f"Inspecting TCIR file (lazy, no full load): {tcir_path}")

    with h5py.File(tcir_path, 'r') as f:
        matrix_shape = f['matrix'].shape      # (N, H, W, C) - metadata only, ~free
        matrix_dtype = f['matrix'].dtype
    print(f"matrix dataset shape: {matrix_shape}, dtype: {matrix_dtype}")

    # 'info' is stored as a pandas table inside the HDF5 file, not a plain
    # h5py group -> must be read with pd.read_hdf, not f['info']['Vmax'][:].
    # It's small (one row of scalars per storm image), so loading it fully is fine.
    print("Loading 'info' table (lightweight metadata, safe to load fully)...")
    info_df = pd.read_hdf(tcir_path, key='info')
    y_tcir_winds_raw = info_df['Vmax'].values.astype(np.float32)

    if np.isnan(y_tcir_winds_raw).any():
        median_wind = np.nanmedian(y_tcir_winds_raw)
        y_tcir_winds_raw = np.nan_to_num(y_tcir_winds_raw, nan=median_wind)
        print(f"NaNs found in wind speeds, replaced with median: {median_wind:.2f}")

    real_num_samples = matrix_shape[0]
    real_channels_in_file = matrix_shape[-1]

    # ---- Compute per-channel min/max by scanning in chunks ----
    # This touches every sample (needed for correct normalization) but only
    # ever holds one chunk in RAM at a time, so it's RAM-safe regardless of
    # how large the full dataset is.
    def compute_channel_min_max(h5_path, chunk=500):
        with h5py.File(h5_path, 'r') as f:
            n, h, w, c = f['matrix'].shape
            mins = np.full(c, np.inf, dtype=np.float32)
            maxs = np.full(c, -np.inf, dtype=np.float32)
            for i in range(0, n, chunk):
                block = f['matrix'][i:i + chunk]
                block = np.nan_to_num(block, nan=0.0)
                mins = np.minimum(mins, block.min(axis=(0, 1, 2)))
                maxs = np.maximum(maxs, block.max(axis=(0, 1, 2)))
                if (i // chunk) % 20 == 0:
                    print(f"  scanned {i + block.shape[0]}/{n} samples for normalization stats...")
        return mins, maxs

    print("Scanning dataset in chunks to compute per-channel min/max (no full load)...")
    channel_min, channel_max = compute_channel_min_max(tcir_path, chunk=500)
    print(f"Per-channel min: {channel_min}")
    print(f"Per-channel max: {channel_max}")

    target_img_size = 64
    target_channels = 4

    print(f"Ready. {real_num_samples} samples available, image shape {matrix_shape[1:]}, "
          f"will resize to {target_img_size}x{target_img_size} and keep {target_channels} channels.")
else:
    raise FileNotFoundError(f"Valid TCIR file path could not be established at: {tcir_path}")

Mounted at /content/drive
Attempting to extract TCIR Dataset from: /content/drive/MyDrive/TCIR-CPAC_IO_SH.7z...
TCIR .7z archive extracted to: /content/tcir_extracted
Inspecting TCIR file (lazy, no full load): /content/tcir_extracted/TCIR-CPAC_IO_SH.h5
matrix dataset shape: (23118, 201, 201, 4), dtype: float32
Loading 'info' table (lightweight metadata, safe to load fully)...
Scanning dataset in chunks to compute per-channel min/max (no full load)...
  scanned 500/23118 samples for normalization stats...
  scanned 10500/23118 samples for normalization stats...
  scanned 20500/23118 samples for normalization stats...
Per-channel min: [ 0.0000000e+00  0.0000000e+00  0.0000000e+00 -9.7075444e-14]
Per-channel max: [3.47820e+02 3.01520e+02 2.20000e+00 9.96921e+36]
Ready. 23118 samples available, image shape (201, 201, 4), will resize to 64x64 and keep 4 channels.


In [ ]:
# ==========================================
# STEP 2.2: LAZY BATCH LOADER (reads from disk on demand, no full-array RAM use)
# ==========================================

class TCIRSequence(tf.keras.utils.Sequence):
    """
    Feeds batches straight from the TCIR .h5 file into model.fit().
    Only ever holds `batch_size` images in memory at once - the rest
    stays on disk until its batch is requested.
    """
    def __init__(self, h5_path, indices, y_all, batch_size=32,
                 target_size=64, target_channels=4,
                 channel_min=None, channel_max=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.h5_path = h5_path
        self.indices = np.array(sorted(indices))  # h5py fancy-indexing needs sorted indices
        self.y_all = y_all
        self.batch_size = batch_size
        self.target_size = target_size
        self.target_channels = target_channels
        self.channel_min = channel_min
        self.channel_max = channel_max
        self.shuffle = shuffle
        self._f = None
        self.on_epoch_end()

    def _file(self):
        # Opened lazily per-worker so it plays nicely with multiprocessing DataLoaders
        if self._f is None:
            self._f = h5py.File(self.h5_path, 'r')
        return self._f

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = np.sort(self.indices[idx * self.batch_size:(idx + 1) * self.batch_size])
        f = self._file()

        X_batch = f['matrix'][batch_idx]          # <-- only this batch touches RAM
        y_batch = self.y_all[batch_idx]

        X_batch = np.nan_to_num(X_batch, nan=0.0).astype(np.float32)

        if self.channel_min is not None:
            for c in range(X_batch.shape[-1]):
                rng = self.channel_max[c] - self.channel_min[c]
                if rng > 0:
                    X_batch[..., c] = (X_batch[..., c] - self.channel_min[c]) / rng
                else:
                    X_batch[..., c] = 0.0

        X_batch = tf.image.resize(X_batch, (self.target_size, self.target_size)).numpy()
        X_batch = X_batch[..., :self.target_channels]

        return X_batch, y_batch

print("TCIRSequence defined - batches will be streamed from disk during training.")


TCIRSequence defined - batches will be streamed from disk during training.


In [ ]:
# ==========================================
# STEP 2.1: IBTRACS DATASET HANDLING (TRACKS)
# ==========================================

import xarray as xr # For NetCDF files

# Mount Google Drive if in Colab to access IBTrACS dataset
if IS_COLAB:
    from google.colab import drive
    # Only mount if not already mounted
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive', force_remount=True)
    ibtracs_path = '/content/drive/MyDrive/IBTrACS.ALL.v04r01.nc'

print(f"Attempting to load IBTrACS Dataset from: {ibtracs_path}...")

try:
    # Open the NetCDF dataset
    ds = xr.open_dataset(ibtracs_path, decode_times=False)
    print("IBTrACS Dataset loaded successfully!")

    # Define feature and target columns
    feature_cols = ['usa_lat', 'usa_lon', 'usa_wind', 'usa_pres']
    target_cols = ['usa_lat', 'usa_lon']

    # Convert to DataFrame and drop rows with any NaN in the relevant columns
    # ibtracs_df has a MultiIndex (storm, date_time). We will group by the 'storm' level.
    ibtracs_df = ds[feature_cols + target_cols].to_dataframe().dropna()

    SEQUENCE_LENGTH = 8 # Matches 'timesteps' in Module 1
    features_list = []
    targets_list = []

    print(f"Creating sequences for storm track prediction (SEQUENCE_LENGTH={SEQUENCE_LENGTH})...")
    for storm_id, group in ibtracs_df.groupby(level='storm'):
        # Extract features and targets as numpy arrays for efficiency
        group_features_values = group[feature_cols].values
        group_targets_values = group[target_cols].values

        # Only create sequences if the storm track is long enough
        if len(group) >= SEQUENCE_LENGTH + 1:
            for i in range(len(group) - SEQUENCE_LENGTH):
                features_list.append(group_features_values[i : i + SEQUENCE_LENGTH])
                targets_list.append(group_targets_values[i + SEQUENCE_LENGTH])

    X_track_real = np.array(features_list).astype(np.float32)
    y_track_real = np.array(targets_list).astype(np.float32)

    # Assign to global variables for use in Module 1 (STEP 3)
    global X_track_real_global, y_track_real_global, real_track_timesteps, real_track_features
    X_track_real_global = X_track_real
    y_track_real_global = y_track_real
    real_track_timesteps = X_track_real.shape[1]
    real_track_features = X_track_real.shape[2]

    print(f"Processed IBTrACS track features shape: {X_track_real.shape}")
    print(f"Processed IBTrACS track targets shape: {y_track_real.shape}")

except FileNotFoundError:
    print(f"Error: IBTrACS dataset not found at {ibtracs_path}. Please check the path and ensure the file is accessible.")
    print("Module 1 will use synthetic data as fallback.")
    X_track_real_global = None
    y_track_real_global = None
except Exception as e:
    print(f"An error occurred while loading or processing the IBTrACS dataset: {e}")
    print("Module 1 will use synthetic data as fallback.")
    X_track_real_global = None
    y_track_real_global = None

Attempting to load IBTrACS Dataset from: /content/drive/MyDrive/IBTrACS.ALL.v04r01.nc...
IBTrACS Dataset loaded successfully!
Creating sequences for storm track prediction (SEQUENCE_LENGTH=8)...
Processed IBTrACS track features shape: (142114, 8, 4)
Processed IBTrACS track targets shape: (142114, 2)


In [ ]:
# ==========================================
# STEP 3: SYNTHETIC DATA & MODEL - MODULE 1
# ==========================================

# 1. Generate Synthetic Data (Shape: [samples, timesteps, features])
# Features: [latitude, longitude, wind_speed, pressure]
num_samples = 1000
timesteps = 8  # Past 24 hours (3-hourly intervals)
features = 4

X_track_synth = np.random.rand(num_samples, timesteps, features)
# Target: Predict next [lat, lon]
y_track_synth = np.random.rand(num_samples, 2)

# Train-Test Split
X_tr_train, X_tr_test, y_tr_train, y_tr_test = train_test_split(X_track_synth, y_track_synth, test_size=0.2)

# 2. Build LSTM Model
track_model = Sequential([
    Input(shape=(timesteps, features)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(2, activation='linear') # Output next lat, lon
])

track_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 3. Train the Model
print("Training Module 1: Track Prediction (LSTM)...")
track_model.fit(X_tr_train, y_tr_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

Training Module 1: Track Prediction (LSTM)...
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.1478 - mae: 0.3149 - val_loss: 0.0825 - val_mae: 0.2448
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0884 - mae: 0.2539 - val_loss: 0.0872 - val_mae: 0.2526
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0872 - mae: 0.2541 - val_loss: 0.0829 - val_mae: 0.2468
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0858 - mae: 0.2527 - val_loss: 0.0889 - val_mae: 0.2532
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0870 - mae: 0.2543 - val_loss: 0.0811 - val_mae: 0.2442


In [ ]:
# ==========================================
# STEP 3.1: EVALUATE MODULE 1 (Track Prediction)
# ==========================================

print("Evaluating Module 1: Track Prediction (LSTM) on test set...")

# Check if real data was used; if so, use the global real data test set if available
if 'X_track_real_global' in globals() and X_track_real_global is not None:
    # Use a train-test split on the global real data
    X_tr_train_real, X_tr_test_real, y_tr_train_real, y_tr_test_real = train_test_split(
        X_track_real_global, y_track_real_global, test_size=0.2, random_state=42
    )
    loss, mae = track_model.evaluate(X_tr_test_real, y_tr_test_real, verbose=0)
    print(f"Module 1 (Real Data) Test Loss (MSE): {loss:.4f}, Test MAE: {mae:.4f}")
else:
    # Fallback to synthetic data test set
    loss, mae = track_model.evaluate(X_tr_test, y_tr_test, verbose=0)
    print(f"Module 1 (Synthetic Data) Test Loss (MSE): {loss:.4f}, Test MAE: {mae:.4f}")

# Make some predictions for demonstration
y_pred_track = track_model.predict(X_tr_test[:5])
print("\nModule 1 Sample Predictions (first 5 test samples):")
for i in range(5):
    print(f"True: {y_tr_test[i]} | Predicted: {y_pred_track[i]}")


Evaluating Module 1: Track Prediction (LSTM) on test set...
Module 1 (Real Data) Test Loss (MSE): 6275.0415, Test MAE: 61.8514
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step

Module 1 Sample Predictions (first 5 test samples):
True: [0.92898394 0.59782172] | Predicted: [0.46420062 0.43650088]
True: [0.36040342 0.64048358] | Predicted: [0.5099748  0.48618168]
True: [0.11916307 0.07209104] | Predicted: [0.4838496 0.455367 ]
True: [0.97280358 0.56258304] | Predicted: [0.41810533 0.3973661 ]
True: [0.07728134 0.0337945 ] | Predicted: [0.53997386 0.5105946 ]


In [ ]:

# ==========================================
# STEP 4: REAL DATA & MODEL - MODULE 2 (Intensity Estimation, memory-safe)
# ==========================================

if 'real_num_samples' not in globals():
    print("Error: TCIR metadata not found. Please run STEP 2 first.")
    print("Falling back to synthetic data for Module 2 to allow continued execution.")
    num_samples = 1000
    img_size = 64
    X_data_for_module2 = np.random.rand(num_samples, img_size, img_size, 4).astype(np.float32)
    y_data_for_module2 = np.random.uniform(30, 150, num_samples).astype(np.float32)
    current_img_size = img_size
    current_channels = 4

    X_im_train, X_im_test, y_im_train, y_im_test = train_test_split(
        X_data_for_module2, y_data_for_module2, test_size=0.2, random_state=42)

    # Assign to X_img_real and y_img_real for consistency with the real data branch
    X_img_real = X_data_for_module2
    y_img_real = y_data_for_module2

    intensity_model = Sequential([
        Input(shape=(current_img_size, current_img_size, current_channels)),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='linear')
    ])
    intensity_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    print("Training Module 2: Intensity Estimation (CNN) with SYNTHETIC data...")
    intensity_model.fit(X_im_train, y_im_train, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

else:
    print(f"Using real TCIR data for Module 2 with {real_num_samples} samples (streamed from disk).")
    current_img_size = target_img_size
    current_channels = target_channels

    all_idx = np.arange(real_num_samples)
    train_idx, test_idx = train_test_split(all_idx, test_size=0.2, random_state=42)

    BATCH_SIZE = 32
    train_gen = TCIRSequence(tcir_path, train_idx, y_tcir_winds_raw,
                              batch_size=BATCH_SIZE, target_size=current_img_size,
                              target_channels=current_channels,
                              channel_min=channel_min, channel_max=channel_max, shuffle=True)
    test_gen = TCIRSequence(tcir_path, test_idx, y_tcir_winds_raw,
                             batch_size=BATCH_SIZE, target_size=current_img_size,
                             target_channels=current_channels,
                             channel_min=channel_min, channel_max=channel_max, shuffle=False)

    # keep names used later in the notebook (STEP 8 save cell references these)
    X_img_real, y_img_real = train_gen, y_tcir_winds_raw

    intensity_model = Sequential([
        Input(shape=(current_img_size, current_img_size, current_channels)),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(1, activation='linear')
    ])
    intensity_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    print("Training Module 2: Intensity Estimation (CNN) with TCIR data (streamed batches)...")
    intensity_model.fit(train_gen, validation_data=test_gen, epochs=5, verbose=1)

Using real TCIR data for Module 2 with 23118 samples (streamed from disk).
Training Module 2: Intensity Estimation (CNN) with TCIR data (streamed batches)...
Epoch 1/5
578/578 ━━━━━━━━━━━━━━━━━━━━ 164s 277ms/step - loss: 781.0693 - mae: 21.2993 - val_loss: 724.3718 - val_mae: 23.3512
Epoch 2/5
578/578 ━━━━━━━━━━━━━━━━━━━━ 156s 269ms/step - loss: 540.6497 - mae: 17.3343 - val_loss: 480.0352 - val_mae: 16.3793
Epoch 3/5
578/578 ━━━━━━━━━━━━━━━━━━━━ 153s 265ms/step - loss: 495.0956 - mae: 16.4531 - val_loss: 465.6559 - val_mae: 16.1920
Epoch 4/5
578/578 ━━━━━━━━━━━━━━━━━━━━ 147s 254ms/step - loss: 480.2292 - mae: 16.1367 - val_loss: 455.2857 - val_mae: 15.3944
Epoch 5/5
578/578 ━━━━━━━━━━━━━━━━━━━━ 148s 255ms/step - loss: 473.6143 - mae: 16.0184 - val_loss: 443.0121 - val_mae: 15.5049


In [ ]:
# ==========================================
# STEP 4.1: EVALUATE MODULE 2 (Intensity Estimation, memory-safe)
# ==========================================

print("Evaluating Module 2: Intensity Estimation (CNN) on test set...")

if 'test_gen' in globals():
    loss, mae = intensity_model.evaluate(test_gen, verbose=0)
    print(f"Module 2 Test Loss (MSE): {loss:.4f}, Test MAE: {mae:.4f}")

    X_sample, y_sample = test_gen[0]  # pull just one batch for a peek, not the whole test set
    y_pred_intensity = intensity_model.predict(X_sample[:5])
    print("\nModule 2 Sample Predictions (first 5 samples of one test batch):")
    for i in range(5):
        print(f"True: {y_sample[i]:.2f} | Predicted: {y_pred_intensity[i][0]:.2f}")
else:
    loss, mae = intensity_model.evaluate(X_im_test, y_im_test, verbose=0)
    print(f"Module 2 Test Loss (MSE): {loss:.4f}, Test MAE: {mae:.4f}")
    y_pred_intensity = intensity_model.predict(X_im_test[:5])
    print("\nModule 2 Sample Predictions (first 5 test samples):")
    for i in range(5):
        print(f"True: {y_im_test[i]:.2f} | Predicted: {y_pred_intensity[i][0]:.2f}")


Evaluating Module 2: Intensity Estimation (CNN) on test set...
Module 2 Test Loss (MSE): 443.0121, Test MAE: 15.5049
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 831ms/step

Module 2 Sample Predictions (first 5 samples of one test batch):
True: 25.00 | Predicted: 32.53
True: 25.00 | Predicted: 28.30
True: 25.00 | Predicted: 41.39
True: 25.00 | Predicted: 30.93
True: 25.00 | Predicted: 29.75


In [ ]:
# ==========================================
# STEP 5: RISK ZONE CLASSIFICATION - MODULE 3
# ==========================================

def classify_risk_zone(predicted_wind, distance_to_coast, vulnerability_index):
    """
    A multi-criteria classification mapping hazard and vulnerability to a zone.
    """
    risk_score = (predicted_wind * 0.5) - (distance_to_coast * 0.3) + (vulnerability_index * 0.2)

    if risk_score > 60:
        return "Red"
    elif risk_score > 30:
        return "Orange"
    else:
        return "Yellow"

# Generate synthetic district DataFrame
districts_df = pd.DataFrame({
    'District_ID': range(1, 101),
    'Distance_to_Path_km': np.random.uniform(10, 300, 100),
    'Predicted_Wind_kt': np.random.uniform(30, 150, 100),
    'Vulnerability_Index': np.random.uniform(10, 100, 100) # Combines exposure/elevation
})

# Apply zoning
districts_df['Risk_Zone'] = districts_df.apply(
    lambda row: classify_risk_zone(row['Predicted_Wind_kt'], row['Distance_to_Path_km'], row['Vulnerability_Index']), axis=1
)

print("\nModule 3 Classification Sample:")
print(districts_df[['District_ID', 'Predicted_Wind_kt', 'Risk_Zone']].head())

In [ ]:
# ==========================================
# STEP 6: SYNTHETIC DATA & MODEL - MODULE 4
# ==========================================

# 1. Prepare Features and Targets
# Features: Wind Speed, Distance, Vulnerability
X_dmg = districts_df[['Predicted_Wind_kt', 'Distance_to_Path_km', 'Vulnerability_Index']]
# Target: Synthetic Economic Damage (in millions USD/INR)
y_dmg = districts_df['Predicted_Wind_kt'] * districts_df['Vulnerability_Index'] * 0.1 + np.random.normal(0, 5, 100)

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(X_dmg, y_dmg, test_size=0.2)

# 2. Build and Train XGBoost Regressor
dmg_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=50, max_depth=3)
dmg_model.fit(X_d_train, y_d_train)

# 3. Predict
districts_df['Estimated_Damage_M'] = dmg_model.predict(X_dmg)
print("\nModule 4 Damage Estimation Sample:")
print(districts_df[['District_ID', 'Risk_Zone', 'Estimated_Damage_M']].head())


Module 4 Damage Estimation Sample:
   District_ID Risk_Zone  Estimated_Damage_M
0            1    Yellow          261.840485
1            2    Yellow         1193.997070
2            3    Orange          910.187134
3            4    Yellow          406.826385
4            5       Red          696.255981


In [ ]:
# ==========================================
# STEP 6.1: EVALUATE MODULE 4 (Damage Estimation)
# ==========================================

from sklearn.metrics import mean_squared_error, mean_absolute_error

print("Evaluating Module 4: Damage Estimation (XGBoost) on test set...")

y_d_pred = dmg_model.predict(X_d_test)
mse = mean_squared_error(y_d_test, y_d_pred)
mae = mean_absolute_error(y_d_test, y_d_pred)

print(f"Module 4 Test Loss (MSE): {mse:.4f}, Test MAE: {mae:.4f}")

# Make some predictions for demonstration
y_pred_damage = dmg_model.predict(X_d_test[:5])
print("\nModule 4 Sample Predictions (first 5 test samples):")
for i in range(5):
    print(f"True: {y_d_test.iloc[i]:.2f} | Predicted: {y_pred_damage[i]:.2f}")


Evaluating Module 4: Damage Estimation (XGBoost) on test set...
Module 4 Test Loss (MSE): 4087.2188, Test MAE: 50.4072

Module 4 Sample Predictions (first 5 test samples):
True: 203.77 | Predicted: 173.53
True: 438.50 | Predicted: 406.83
True: 230.50 | Predicted: 191.91
True: 436.26 | Predicted: 485.51
True: 608.31 | Predicted: 585.49


In [ ]:
# ==========================================
# STEP 7: ALERT DISPATCH GENERATOR - MODULE 5
# ==========================================

def generate_cap_alert(district_data):
    alerts = []
    for _, row in district_data.iterrows():
        zone = row['Risk_Zone']
        if zone == "Red":
            urgency = "Immediate"
            instruction = "Evacuation advisory in effect. Activate emergency shelters immediately."
        elif zone == "Orange":
            urgency = "Expected"
            instruction = "Prepare to evacuate. Secure coastal properties and stage resources."
        else:
            urgency = "Future"
            instruction = "Monitor local weather stations for precautionary advisories."

        alert_json = {
            "identifier": f"ALERT-DIST-{int(row['District_ID'])}",
            "info": {
                "category": "Met",
                "event": "Tropical Cyclone",
                "urgency": urgency,
                "severity": zone,
                "description": f"Predicted wind speeds up to {row['Predicted_Wind_kt']:.1f} kt.",
                "instruction": instruction
            }
        }
        alerts.append(alert_json)
    return alerts

generated_alerts = generate_cap_alert(districts_df.head(3))
print("\nModule 5 Automated Alert Sample (CAP Format):")
print(json.dumps(generated_alerts, indent=2))


Module 5 Automated Alert Sample (CAP Format):
[
  {
    "identifier": "ALERT-DIST-1",
    "info": {
      "category": "Met",
      "event": "Tropical Cyclone",
      "urgency": "Future",
      "severity": "Yellow",
      "description": "Predicted wind speeds up to 36.3 kt.",
      "instruction": "Monitor local weather stations for precautionary advisories."
    }
  },
  {
    "identifier": "ALERT-DIST-2",
    "info": {
      "category": "Met",
      "event": "Tropical Cyclone",
      "urgency": "Future",
      "severity": "Yellow",
      "description": "Predicted wind speeds up to 125.5 kt.",
      "instruction": "Monitor local weather stations for precautionary advisories."
    }
  },
  {
    "identifier": "ALERT-DIST-3",
    "info": {
      "category": "Met",
      "event": "Tropical Cyclone",
      "urgency": "Expected",
      "severity": "Orange",
      "description": "Predicted wind speeds up to 122.1 kt.",
      "instruction": "Prepare to evacuate. Secure coastal properties and 

In [ ]:
# ==========================================
# STEP 8: MAKE DATASETS READY FOR DOWNLOAD
# ==========================================

print("Saving datasets locally...")

# Fallback for X_track_synth and y_track_synth if Module 1 was not executed or its state was lost
if 'X_track_synth' not in globals() or 'y_track_synth' not in globals():
    print("Warning: X_track_synth or y_track_synth not found. Generating minimal synthetic data for saving.")
    num_samples_fallback = 100
    timesteps_fallback = 8
    features_fallback = 4
    X_track_synth = np.random.rand(num_samples_fallback, timesteps_fallback, features_fallback).astype(np.float32)
    y_track_synth = np.random.rand(num_samples_fallback, 2).astype(np.float32)

# 1. Save Tabular Sequence Data (Track Prediction)
# These are generated as synthetic in this notebook.
np.save('synthetic_track_features.npy', X_track_synth)
np.save('synthetic_track_targets.npy', y_track_synth)
print("Saved: synthetic_track_features.npy, synthetic_track_targets.npy")

# Determine if real TCIR data was used for Intensity Estimation (Module 2)
is_tcir_real_data_used = 'real_num_samples' in globals() and real_num_samples > 0

# 2. Save Image Tensors and Winds (TCIR - Intensity Estimation)
if is_tcir_real_data_used:
    print("\nReal TCIR data was used for Intensity Estimation.")
    # X_img_real is a TCIRSequence generator, designed for memory-safe streaming.
    # Saving the entire image dataset as a single .npy would defeat this and likely cause OOM.
    print("Note: The real TCIR image data (X_img_real) is streamed from HDF5 and is not saved as a single .npy file to prevent memory overload.")
    # y_img_real (y_tcir_winds_raw) is a NumPy array of real wind speeds and can be saved.
    np.save('real_tcir_winds.npy', y_img_real)
    print("Saved: real_tcir_winds.npy (raw wind speeds for real TCIR data)")
else:
    print("\nSynthetic data was used for Intensity Estimation.")
    # X_img_real and y_img_real will be NumPy arrays of synthetic data in this case.
    np.save('synthetic_tcir_images.npy', X_img_real)
    np.save('synthetic_tcir_winds.npy', y_img_real)
    print("Saved: synthetic_tcir_images.npy, synthetic_tcir_winds.npy")

# Fallback for districts_df if Module 3 was not executed or its state was lost
if 'districts_df' not in globals():
    print("Warning: districts_df not found. Generating minimal synthetic DataFrame for saving.")
    districts_df = pd.DataFrame({
        'District_ID': range(1, 11),
        'Distance_to_Path_km': np.random.uniform(10, 300, 10),
        'Predicted_Wind_kt': np.random.uniform(30, 150, 10),
        'Vulnerability_Index': np.random.uniform(10, 100, 10)
    })
    districts_df['Risk_Zone'] = np.random.choice(['Red', 'Orange', 'Yellow'], 10)

# 3. Save GIS/Zoning and Damage DataFrame (always synthetic in this notebook)
districts_df.to_csv('synthetic_district_risk.csv', index=False)
print("Saved: synthetic_district_risk.csv")

print("\nAll specified datasets are processed for local storage!")

Saving datasets locally...
Saved: synthetic_track_features.npy, synthetic_track_targets.npy

Real TCIR data was used for Intensity Estimation.
Note: The real TCIR image data (X_img_real) is streamed from HDF5 and is not saved as a single .npy file to prevent memory overload.
Saved: real_tcir_winds.npy (raw wind speeds for real TCIR data)
Saved: synthetic_district_risk.csv

All specified datasets are processed for local storage!


In [ ]:
if IS_COLAB:
    from google.colab import files
    import os # Import os to check file existence
    print("\nEnabling file downloads for Colab...")

    # Always download track and district data (synthetic)
    files.download('synthetic_track_features.npy')
    files.download('synthetic_track_targets.npy')
    files.download('synthetic_district_risk.csv')
    print("Downloading synthetic_track_features.npy, synthetic_track_targets.npy, synthetic_district_risk.csv")

    # Conditionally download TCIR image/wind data
    is_tcir_real_data_used = 'real_num_samples' in globals() and real_num_samples > 0

    if is_tcir_real_data_used:
        # Only real_tcir_winds.npy was saved as a numpy array
        if os.path.exists('real_tcir_winds.npy'):
            files.download('real_tcir_winds.npy')
            print("Downloading real_tcir_winds.npy")
        else:
            print("Note: real_tcir_winds.npy not found for download.")
    else:
        # Synthetic TCIR images and winds were saved
        if os.path.exists('synthetic_tcir_images.npy'):
            files.download('synthetic_tcir_images.npy')
            print("Downloading synthetic_tcir_images.npy")
        else:
            print("Note: synthetic_tcir_images.npy not found for download.")
        if os.path.exists('synthetic_tcir_winds.npy'):
            files.download('synthetic_tcir_winds.npy')
            print("Downloading synthetic_tcir_winds.npy")
        else:
            print("Note: synthetic_tcir_winds.npy not found for download.")

    print("\nDownloads initiated!")


Enabling file downloads for Colab...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloads initiated!


In [ ]:
print("Saving trained models locally...")

# Save Track Prediction (LSTM) Model
track_model.save('track_prediction_lstm_model.h5')

# Save Intensity Estimation (CNN) Model
intensity_model.save('intensity_estimation_cnn_model.h5')

# Save Damage Estimation (XGBoost) Model
dmg_model.save_model('damage_estimation_xgb_model.json')

print("Models successfully saved to Colab storage!")

Saving trained models locally...


Models successfully saved to Colab storage!


In [ ]:
# Uncomment below to trigger automated browser downloads:
# files.download('track_prediction_lstm_model.h5') # Commented out for Jupyter compatibility
# files.download('intensity_estimation_cnn_model.h5') # Commented out for Jupyter compatibility
# files.download('damage_estimation_xgb_model.json') # Commented out for Jupyter compatibility

In [ ]:
if IS_COLAB:
    from google.colab import files
    # Uncomment below to trigger automated browser downloads:
    files.download('track_prediction_lstm_model.h5')
    files.download('intensity_estimation_cnn_model.h5')
    files.download('damage_estimation_xgb_model.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>